In [1]:
import pandas as pd
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
df = pd.read_csv('finaldataset_with_sets.csv')  

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(text):
    return tokenizer(text, padding='max_length', truncation=True, return_tensors="pt")

df['input_ids'] = df['transcript'].apply(lambda x: tokenize_function(x)['input_ids'].squeeze())
df['attention_mask'] = df['transcript'].apply(lambda x: tokenize_function(x)['attention_mask'].squeeze())

class TextDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        input_ids = self.dataframe.iloc[idx]['input_ids']
        attention_mask = self.dataframe.iloc[idx]['attention_mask']
        label = self.dataframe.iloc[idx]['label']
        return input_ids, attention_mask, label

train_df = df[df['set'] == 'training']
val_df = df[df['set'] == 'validation']

train_dataset = TextDataset(train_df)
val_dataset = TextDataset(val_df)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=16)


/opt/anaconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
class TextClassificationModel(nn.Module):
    def __init__(self, num_classes=5, dropout_rate=0.5):
        super(TextClassificationModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs[1]  # CLS token
        dropout_output = self.dropout(pooled_output)
        return self.fc(dropout_output)

# Initialize model
text_model = TextClassificationModel(num_classes=5, dropout_rate=0.5).to(device)

# Optimizer and Scheduler
optimizer = AdamW(text_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_dataloader) * 10  # Adjust for the number of epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# Class Weights
labels = train_df['label'].values
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Loss Function
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, logits=False, reduce=True):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.logits = logits
        self.reduce = reduce

    def forward(self, inputs, targets):
        BCE_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss

        if self.reduce:
            return torch.mean(F_loss)
        else:
            return F_loss

loss_fn = FocalLoss(logits=True).to(device)

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


### Training

In [ ]:
def train_model(model, train_dataloader, val_dataloader, optimizer, scheduler, loss_fn, num_epochs=10, patience=3):
    best_val_f1 = 0
    patience_counter = 0
    best_model_state_dict = None

    for epoch in range(num_epochs):
        model.train()
        total_loss, total_correct = 0, 0
        total_preds, total_labels = [], []

        for batch in train_dataloader:
            optimizer.zero_grad()
            input_ids, attention_mask, labels = batch[0].to(device), batch[1].to(device), batch[2].to(device)
            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            total_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            total_labels.extend(labels.cpu().numpy())

        train_accuracy = accuracy_score(total_labels, total_preds)
        train_f1 = precision_recall_fscore_support(total_labels, total_preds, average='weighted')[2]

        val_loss, val_accuracy, val_f1 = evaluate_model(model, val_dataloader, loss_fn)
        print(f'Epoch {epoch+1}/{num_epochs} - Loss: {total_loss/len(train_dataloader):.4f} - '
              f'Accuracy: {train_accuracy:.4f} - F1 Score: {train_f1:.4f}')
        print(f'Validation - Loss: {val_loss:.4f} - Accuracy: {val_accuracy:.4f} - F1 Score: {val_f1:.4f}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state_dict = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    # Load best model weights
    if best_model_state_dict is not None:
        model.load_state_dict(best_model_state_dict)
    return model

# Evaluation function
def evaluate_model(model, val_dataloader, loss_fn):
    model.eval()
    total_loss, total_correct = 0, 0
    total_preds, total_labels = [], []

    with torch.no_grad():
        for batch in val_dataloader:
            input_ids, attention_mask, labels = batch[0].to(device), batch[1].to(device), batch[2].to(device)
            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()

            total_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            total_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(total_labels, total_preds)
    f1 = precision_recall_fscore_support(total_labels, total_preds, average='weighted')[2]
    return total_loss / len(val_dataloader), accuracy, f1

# Train the model
text_model = train_model(text_model, train_dataloader, val_dataloader, optimizer, scheduler, loss_fn, num_epochs=10, patience=3)

# Save the best model
torch.save(text_model.state_dict(), 'text_model_final3.pth')


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/10 - Loss: 0.8938 - Accuracy: 0.3875 - F1 Score: 0.3651
Validation - Loss: 0.6557 - Accuracy: 0.5500 - F1 Score: 0.3903


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2/10 - Loss: 0.7632 - Accuracy: 0.4575 - F1 Score: 0.3858
Validation - Loss: 0.6497 - Accuracy: 0.5500 - F1 Score: 0.3903


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 3/10 - Loss: 0.7549 - Accuracy: 0.4625 - F1 Score: 0.3866
Validation - Loss: 0.6613 - Accuracy: 0.5500 - F1 Score: 0.3903


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 4/10 - Loss: 0.7098 - Accuracy: 0.4825 - F1 Score: 0.4001
Validation - Loss: 0.6551 - Accuracy: 0.5500 - F1 Score: 0.3903
Early stopping triggered.
